# Pure BO vs Pure DR (Gym Surrogate -> TinySim Target)

This notebook compares two baselines for sim-to-sim transfer to TinySim MountainCar target params.

- **Pure DR**: train in Gym with uniform domain randomization over `(force, gravity)`
- **Pure BO**: use GP BO over Gym center `(force, gravity)`, train at fixed center each BO step (no DR adaptation)

Both use your MountainCar DQN checkpoint as warm start.

## 1) Imports and Global Setup

In [ ]:
from __future__ import annotations

import copy
import csv
import json
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import gymnasium as gym
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import HTML, display

from tinysim.mountain_car import MountainCarEnv

In [ ]:
# BO dependency check
try:
    from skopt import Optimizer
    from skopt.space import Real
except Exception as exc:
    raise ImportError('scikit-optimize is required: pip install scikit-optimize') from exc

print('skopt imported successfully')

## 2) Config (edit this cell)

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)

PROJECT_DIR = Path('/home/buchkeva/IdeaTesting/TinySim')
RUNS_ROOT = PROJECT_DIR / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

WARMSTART_CHECKPOINT = PROJECT_DIR / 'mountain_car_saved' / 'mountain_car_dqn.pt'

# TinySim target (evaluation only) - same hard edge case as BO+DORAEMON notebook
TARGET_PARAMS = {
    'force': 0.0010,
    'gravity': 0.0050,
}

# Surrogate ranges
DR_BOUNDS = {
    'force': (0.0003, 0.0030),
    'gravity': (0.0010, 0.0062),
}

# Focused edge bounds (same as BO+DORAEMON EDGE_DR_BOUNDS)
EDGE_BOUNDS = {
    'force': (0.0007, 0.0018),
    'gravity': (0.0030, 0.0059),
}

# Use focused edge bounds by default for both Pure DR and Pure BO.
TRAIN_BOUNDS = EDGE_BOUNDS
BO_BOUNDS = EDGE_BOUNDS

MU0 = np.array([0.0010, 0.0025], dtype=np.float32)
MU0_EDGE = np.array([
    float(np.clip(MU0[0], EDGE_BOUNDS['force'][0], EDGE_BOUNDS['force'][1])),
    float(np.clip(MU0[1], EDGE_BOUNDS['gravity'][0], EDGE_BOUNDS['gravity'][1])),
], dtype=np.float32)

TAU_STEPS = 200
SUCCESS_THRESHOLD = 0.95

# Match BO+DORAEMON budgets: T = bo_iters, K = bo_train_episodes_per_iter, B_r = target_eval_episodes
SMOKE_CFG = {
    'bo_iters': 3,
    'bo_train_episodes_per_iter': 20,
    'dr_total_episodes': 60,
    'dr_eval_every': 20,
    'target_eval_episodes': 5,
}

PROTOTYPE_CFG = {
    'bo_iters': 12,
    'bo_train_episodes_per_iter': 120,
    'dr_total_episodes': 1440,
    'dr_eval_every': 120,
    'target_eval_episodes': 12,
}

EDGE_CFG = {
    'bo_iters': 20,
    'bo_train_episodes_per_iter': 180,
    'dr_total_episodes': 3600,
    'dr_eval_every': 180,
    'target_eval_episodes': 20,
}

for _cfg in [SMOKE_CFG, PROTOTYPE_CFG, EDGE_CFG]:
    assert _cfg['dr_total_episodes'] == _cfg['bo_iters'] * _cfg['bo_train_episodes_per_iter']

print('Warmstart:', WARMSTART_CHECKPOINT)
print('Target params:', TARGET_PARAMS)
print('TRAIN bounds:', TRAIN_BOUNDS)
print('BO bounds:', BO_BOUNDS)
print('MU0_EDGE:', MU0_EDGE.tolist())

## 3) DQN Components

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int = 10000):
        self.capacity = int(capacity)
        self.buffer = []
        self.pos = 0

    def push(self, state, action, reward, next_state, done):
        item = (
            np.asarray(state, dtype=np.float32),
            int(action),
            float(reward),
            np.asarray(next_state, dtype=np.float32),
            float(done),
        )
        if len(self.buffer) < self.capacity:
            self.buffer.append(item)
        else:
            self.buffer[self.pos] = item
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size: int, device: str):
        idx = np.random.choice(len(self.buffer), size=batch_size, replace=False)
        batch = [self.buffer[i] for i in idx]
        s, a, r, ns, d = zip(*batch)
        states = torch.tensor(np.stack(s), dtype=torch.float32, device=device)
        actions = torch.tensor(a, dtype=torch.long, device=device)
        rewards = torch.tensor(r, dtype=torch.float32, device=device)
        next_states = torch.tensor(np.stack(ns), dtype=torch.float32, device=device)
        dones = torch.tensor(d, dtype=torch.float32, device=device)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)


class QNetwork(nn.Module):
    def __init__(self, state_dim: int = 2, n_actions: int = 3, hidden_dim: int = 128):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


@dataclass
class DQNConfig:
    state_dim: int = 2
    n_actions: int = 3
    hidden_dim: int = 128
    lr: float = 1e-3
    gamma: float = 0.99
    epsilon_start: float = 1.0
    epsilon_end: float = 0.01
    epsilon_decay: float = 0.995
    buffer_capacity: int = 10000
    batch_size: int = 64
    target_update_freq: int = 10


class DQNAgent:
    def __init__(self, cfg: DQNConfig, device: str = 'cpu'):
        self.cfg = cfg
        self.device = device

        self.policy_net = QNetwork(cfg.state_dim, cfg.n_actions, cfg.hidden_dim).to(device)
        self.target_net = QNetwork(cfg.state_dim, cfg.n_actions, cfg.hidden_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=cfg.lr)
        self.memory = ReplayBuffer(cfg.buffer_capacity)

        self.epsilon = cfg.epsilon_start
        self.training_losses = []
        self._episode_counter = 0

    def act(self, state: np.ndarray, training: bool = True) -> int:
        if training and random.random() < self.epsilon:
            return random.randrange(self.cfg.n_actions)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            q = self.policy_net(s)
            return int(q.argmax(dim=1).item())

    def store(self, s, a, r, ns, done):
        self.memory.push(s, a, r, ns, done)

    def learn_step(self):
        if len(self.memory) < self.cfg.batch_size:
            return None
        s, a, r, ns, d = self.memory.sample(self.cfg.batch_size, self.device)

        q = self.policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            nq = self.target_net(ns).max(dim=1)[0]
            target = r + self.cfg.gamma * nq * (1.0 - d)

        loss = nn.MSELoss()(q, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        self.training_losses.append(float(loss.item()))
        return float(loss.item())

    def end_episode(self):
        self._episode_counter += 1
        self.epsilon = max(self.cfg.epsilon_end, self.epsilon * self.cfg.epsilon_decay)
        if (self._episode_counter % self.cfg.target_update_freq) == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

    def state_payload(self) -> dict[str, Any]:
        return {
            'policy_state_dict': self.policy_net.state_dict(),
            'target_state_dict': self.target_net.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'epsilon': self.epsilon,
            'training_losses': self.training_losses,
            'hparams': vars(self.cfg),
        }

    def load_payload(self, payload: dict[str, Any]):
        self.policy_net.load_state_dict(payload['policy_state_dict'])
        self.target_net.load_state_dict(payload.get('target_state_dict', payload['policy_state_dict']))
        if 'optimizer_state_dict' in payload:
            self.optimizer.load_state_dict(payload['optimizer_state_dict'])
        self.epsilon = float(payload.get('epsilon', self.cfg.epsilon_end))
        self.training_losses = list(payload.get('training_losses', []))

## 4) Checkpoint and Env Helpers

In [ ]:
def make_agent_from_hparams(hparams: dict | None, device: str = DEVICE) -> DQNAgent:
    if hparams is None:
        cfg = DQNConfig()
    else:
        filtered = {k: v for k, v in hparams.items() if k in DQNConfig.__annotations__}
        cfg = DQNConfig(**filtered)
    return DQNAgent(cfg=cfg, device=device)


def load_warmstart_agent(checkpoint_path: str | Path, device: str = DEVICE) -> DQNAgent:
    payload = torch.load(str(checkpoint_path), map_location=device)
    agent = make_agent_from_hparams(payload.get('hparams'), device=device)
    agent.load_payload(payload)
    return agent


def save_agent_checkpoint(agent: DQNAgent, path: str | Path, extra: dict | None = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = agent.state_payload()
    payload['extra'] = extra or {}
    torch.save(payload, str(path))


def project_mu_to_bounds(mu: np.ndarray, bounds: dict) -> np.ndarray:
    return np.array([
        float(np.clip(mu[0], bounds['force'][0], bounds['force'][1])),
        float(np.clip(mu[1], bounds['gravity'][0], bounds['gravity'][1])),
    ], dtype=np.float32)


def make_gym_env(force: float, gravity: float, seed: int | None = None):
    env = gym.make('MountainCar-v0')
    env.reset(seed=seed)
    env.unwrapped.force = float(force)
    env.unwrapped.gravity = float(gravity)
    return env


def sample_uniform_params(bounds: dict, rng: np.random.Generator) -> tuple[float, float]:
    force = float(rng.uniform(bounds['force'][0], bounds['force'][1]))
    gravity = float(rng.uniform(bounds['gravity'][0], bounds['gravity'][1]))
    return force, gravity


def run_train_episode(agent: DQNAgent, force: float, gravity: float, tau_steps: int, seed: int) -> dict:
    env = make_gym_env(force=force, gravity=gravity, seed=seed)
    s, _ = env.reset(seed=seed)
    losses = []
    solved = False
    steps = 0

    for _ in range(tau_steps):
        a = agent.act(s, training=True)
        ns, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        agent.store(s, a, r, ns, float(done))
        loss = agent.learn_step()
        if loss is not None:
            losses.append(loss)
        s = ns
        steps += 1
        if terminated:
            solved = True
            break
        if truncated:
            break

    env.close()
    agent.end_episode()

    return {
        'solved': bool(solved),
        'steps': int(steps),
        'loss_mean': float(np.mean(losses)) if losses else np.nan,
    }


def target_eval_tinysim_detailed(
    agent: DQNAgent,
    n_eval_target: int,
    max_steps: int,
    target_params: dict,
) -> tuple[float, int, int]:
    successes = 0

    for _ in range(n_eval_target):
        sim_env = MountainCarEnv()
        sim_env.force = float(target_params['force'])
        sim_env.gravity = float(target_params['gravity'])

        state = sim_env.reset()
        obs = np.array([state['position'], state['velocity']], dtype=np.float32)

        solved = False
        for _ in range(max_steps):
            a = agent.act(obs, training=False)
            state = sim_env.step(a)
            obs = np.array([state['position'], state['velocity']], dtype=np.float32)
            if bool(state['done']):
                solved = True
                break

        successes += int(solved)

    rate = float(successes / max(1, n_eval_target))
    return rate, int(successes), int(n_eval_target)

## 5) Pure DR Baseline

In [ ]:
def run_pure_dr(
    checkpoint_path: str | Path,
    total_episodes: int,
    eval_every: int,
    target_eval_episodes: int,
    tau_steps: int,
    bounds: dict,
    target_params: dict,
    seed: int = 42,
) -> dict:
    run_name = time.strftime('bo_vs_dr_pure_dr_%Y%m%d_%H%M%S')
    run_dir = RUNS_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    history_json = run_dir / 'history.json'
    history_csv = run_dir / 'history.csv'
    best_ckpt = run_dir / 'best_policy.pt'
    last_ckpt = run_dir / 'last_policy.pt'

    rng = np.random.default_rng(seed)
    agent = load_warmstart_agent(checkpoint_path=checkpoint_path, device=DEVICE)

    t0 = time.perf_counter()

    f0, succ0, eps0 = target_eval_tinysim_detailed(agent, target_eval_episodes, tau_steps, target_params)
    best_score = float(f0)
    save_agent_checkpoint(agent, best_ckpt, extra={'episode': 0, 'score': best_score})

    history = [{
        'method': 'pure_dr',
        'episode': 0,
        'train_episodes': 0,
        'elapsed_sec': 0.0,
        'force': float('nan'),
        'gravity': float('nan'),
        'loss_mean': float('nan'),
        'target_solve_rate': float(f0),
        'target_successes': int(succ0),
        'target_episodes': int(eps0),
        'is_best': True,
    }]

    for ep in range(1, total_episodes + 1):
        force, gravity = sample_uniform_params(bounds, rng)
        train_info = run_train_episode(agent, force, gravity, tau_steps=tau_steps, seed=seed + ep)

        if (ep % eval_every == 0) or (ep == total_episodes):
            f_t, succ_t, eps_t = target_eval_tinysim_detailed(
                agent, target_eval_episodes, tau_steps, target_params
            )

            is_best = False
            if float(f_t) > best_score:
                best_score = float(f_t)
                save_agent_checkpoint(agent, best_ckpt, extra={'episode': ep, 'score': best_score})
                is_best = True

            row = {
                'method': 'pure_dr',
                'episode': int(ep),
                'train_episodes': int(ep),
                'elapsed_sec': float(time.perf_counter() - t0),
                'force': float(force),
                'gravity': float(gravity),
                'loss_mean': float(train_info['loss_mean']) if not np.isnan(train_info['loss_mean']) else np.nan,
                'target_solve_rate': float(f_t),
                'target_successes': int(succ_t),
                'target_episodes': int(eps_t),
                'is_best': bool(is_best),
            }
            history.append(row)

            print(f"[Pure DR] ep={ep}/{total_episodes} target={succ_t}/{eps_t} rate={f_t:.3f}")

            with open(history_json, 'w') as f:
                json.dump(history, f, indent=2)
            with open(history_csv, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
                writer.writeheader()
                writer.writerows(history)

    save_agent_checkpoint(agent, last_ckpt, extra={'episode': total_episodes, 'best_score': best_score})

    return {
        'method': 'pure_dr',
        'run_dir': str(run_dir),
        'history': history,
        'history_json': str(history_json),
        'history_csv': str(history_csv),
        'best_checkpoint': str(best_ckpt),
        'last_checkpoint': str(last_ckpt),
        'best_score': float(best_score),
        'elapsed_sec': float(time.perf_counter() - t0),
    }

## 6) Pure BO Baseline (No DR)

In [ ]:
def run_pure_bo(
    checkpoint_path: str | Path,
    mu0: np.ndarray,
    bounds: dict,
    bo_iters: int,
    train_episodes_per_iter: int,
    target_eval_episodes: int,
    tau_steps: int,
    target_params: dict,
    seed: int = 42,
    epsilon_floor_each_iter: float = 0.10,
) -> dict:
    run_name = time.strftime('bo_vs_dr_pure_bo_%Y%m%d_%H%M%S')
    run_dir = RUNS_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    history_json = run_dir / 'history.json'
    history_csv = run_dir / 'history.csv'
    best_ckpt = run_dir / 'best_policy.pt'
    last_ckpt = run_dir / 'last_policy.pt'

    agent = load_warmstart_agent(checkpoint_path=checkpoint_path, device=DEVICE)

    optimizer = Optimizer(
        dimensions=[
            Real(bounds['force'][0], bounds['force'][1], name='force_center'),
            Real(bounds['gravity'][0], bounds['gravity'][1], name='gravity_center'),
        ],
        base_estimator='GP',
        acq_func='EI',
        random_state=seed,
    )

    mu0_proj = project_mu_to_bounds(np.array(mu0, dtype=np.float32), bounds)
    if not np.allclose(mu0_proj, mu0):
        print(f"[Pure BO init] mu0 projected to {mu0_proj.tolist()}")

    t0 = time.perf_counter()

    f0, succ0, eps0 = target_eval_tinysim_detailed(agent, target_eval_episodes, tau_steps, target_params)
    optimizer.tell(mu0_proj.tolist(), -float(f0))

    best_score = float(f0)
    best_center = mu0_proj.copy()
    save_agent_checkpoint(agent, best_ckpt, extra={'iter': 0, 'center': mu0_proj.tolist(), 'score': best_score})

    history = [{
        'method': 'pure_bo',
        'iter': 0,
        'train_episodes': 0,
        'elapsed_sec': 0.0,
        'mu_force': float(mu0_proj[0]),
        'mu_gravity': float(mu0_proj[1]),
        'loss_mean': float('nan'),
        'target_solve_rate': float(f0),
        'target_successes': int(succ0),
        'target_episodes': int(eps0),
        'is_best': True,
    }]

    cumulative_episodes = 0

    for t in range(1, bo_iters + 1):
        mu_t = np.array(optimizer.ask(), dtype=np.float32)
        agent.epsilon = max(float(agent.epsilon), float(epsilon_floor_each_iter))

        losses = []
        for ep in range(train_episodes_per_iter):
            info = run_train_episode(
                agent,
                force=float(mu_t[0]),
                gravity=float(mu_t[1]),
                tau_steps=tau_steps,
                seed=seed + 10000 * t + ep,
            )
            if not np.isnan(info['loss_mean']):
                losses.append(float(info['loss_mean']))

        cumulative_episodes += int(train_episodes_per_iter)

        f_t, succ_t, eps_t = target_eval_tinysim_detailed(agent, target_eval_episodes, tau_steps, target_params)
        optimizer.tell(mu_t.tolist(), -float(f_t))

        is_best = False
        if float(f_t) > best_score:
            best_score = float(f_t)
            best_center = mu_t.copy()
            save_agent_checkpoint(
                agent,
                best_ckpt,
                extra={'iter': t, 'center': mu_t.tolist(), 'score': best_score},
            )
            is_best = True

        row = {
            'method': 'pure_bo',
            'iter': int(t),
            'train_episodes': int(cumulative_episodes),
            'elapsed_sec': float(time.perf_counter() - t0),
            'mu_force': float(mu_t[0]),
            'mu_gravity': float(mu_t[1]),
            'loss_mean': float(np.mean(losses)) if losses else np.nan,
            'target_solve_rate': float(f_t),
            'target_successes': int(succ_t),
            'target_episodes': int(eps_t),
            'is_best': bool(is_best),
        }
        history.append(row)

        print(
            f"[Pure BO] iter={t}/{bo_iters} mu=({mu_t[0]:.6f}, {mu_t[1]:.6f}) "
            f"target={succ_t}/{eps_t} rate={f_t:.3f}"
        )

        with open(history_json, 'w') as f:
            json.dump(history, f, indent=2)
        with open(history_csv, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
            writer.writeheader()
            writer.writerows(history)

    save_agent_checkpoint(agent, last_ckpt, extra={'iter': bo_iters, 'best_score': best_score})

    return {
        'method': 'pure_bo',
        'run_dir': str(run_dir),
        'history': history,
        'history_json': str(history_json),
        'history_csv': str(history_csv),
        'best_checkpoint': str(best_ckpt),
        'last_checkpoint': str(last_ckpt),
        'best_center': best_center.tolist(),
        'best_score': float(best_score),
        'elapsed_sec': float(time.perf_counter() - t0),
    }

## 7) Run Comparison

In [ ]:
assert WARMSTART_CHECKPOINT.exists(), f'Missing warmstart checkpoint: {WARMSTART_CHECKPOINT}'

# Choose run profile to match BO+DORAEMON presets: 'smoke' | 'prototype' | 'edge'
RUN_PROFILE = 'edge'
if RUN_PROFILE == 'smoke':
    cfg = SMOKE_CFG
elif RUN_PROFILE == 'prototype':
    cfg = PROTOTYPE_CFG
elif RUN_PROFILE == 'edge':
    cfg = EDGE_CFG
else:
    raise ValueError(f'Unknown RUN_PROFILE: {RUN_PROFILE}')

print('Run profile:', RUN_PROFILE)
print('Using config:', cfg)

dr_result = run_pure_dr(
    checkpoint_path=WARMSTART_CHECKPOINT,
    total_episodes=cfg['dr_total_episodes'],
    eval_every=cfg['dr_eval_every'],
    target_eval_episodes=cfg['target_eval_episodes'],
    tau_steps=TAU_STEPS,
    bounds=TRAIN_BOUNDS,
    target_params=TARGET_PARAMS,
    seed=SEED,
)

bo_result = run_pure_bo(
    checkpoint_path=WARMSTART_CHECKPOINT,
    mu0=MU0_EDGE,
    bounds=BO_BOUNDS,
    bo_iters=cfg['bo_iters'],
    train_episodes_per_iter=cfg['bo_train_episodes_per_iter'],
    target_eval_episodes=cfg['target_eval_episodes'],
    tau_steps=TAU_STEPS,
    target_params=TARGET_PARAMS,
    seed=SEED + 123,
)

print('DR run_dir:', dr_result['run_dir'])
print('BO run_dir:', bo_result['run_dir'])

## 8) Summary Table and Feasibility

In [ ]:
def first_threshold_step(history: list[dict], threshold: float, key_step: str) -> int | None:
    for r in history:
        if float(r['target_solve_rate']) >= threshold:
            return int(r[key_step])
    return None

dr_first = first_threshold_step(dr_result['history'], SUCCESS_THRESHOLD, 'train_episodes')
bo_first = first_threshold_step(bo_result['history'], SUCCESS_THRESHOLD, 'train_episodes')

summary = pd.DataFrame([
    {
        'method': 'Pure DR',
        'best_target_solve_rate': dr_result['best_score'],
        'elapsed_sec': dr_result['elapsed_sec'],
        'reached_threshold': dr_first is not None,
        'episodes_to_threshold': dr_first,
        'best_checkpoint': dr_result['best_checkpoint'],
        'run_dir': dr_result['run_dir'],
    },
    {
        'method': 'Pure BO',
        'best_target_solve_rate': bo_result['best_score'],
        'elapsed_sec': bo_result['elapsed_sec'],
        'reached_threshold': bo_first is not None,
        'episodes_to_threshold': bo_first,
        'best_checkpoint': bo_result['best_checkpoint'],
        'run_dir': bo_result['run_dir'],
    },
])

display(summary)

## 9) Progress Plots

In [ ]:
dr_hist = pd.DataFrame(dr_result['history'])
bo_hist = pd.DataFrame(bo_result['history'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(dr_hist['train_episodes'], dr_hist['target_solve_rate'], marker='o', label='Pure DR')
axes[0].plot(bo_hist['train_episodes'], bo_hist['target_solve_rate'], marker='o', label='Pure BO')
axes[0].set_xlabel('Training Episodes')
axes[0].set_ylabel('TinySim Solve Rate')
axes[0].set_title('Solve Rate vs Episodes')
axes[0].set_ylim(-0.02, 1.02)
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(dr_hist['elapsed_sec'], dr_hist['target_solve_rate'], marker='o', label='Pure DR')
axes[1].plot(bo_hist['elapsed_sec'], bo_hist['target_solve_rate'], marker='o', label='Pure BO')
axes[1].set_xlabel('Elapsed Seconds')
axes[1].set_ylabel('TinySim Solve Rate')
axes[1].set_title('Solve Rate vs Wall Time')
axes[1].set_ylim(-0.02, 1.02)
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 10) Optional Playback of Best Policies

In [ ]:
def render_tinysim_state_frame(position, min_position=-1.2, max_position=0.6, goal_position=0.5):
    fig, ax = plt.subplots(figsize=(6, 4), dpi=100)
    x = np.linspace(min_position, max_position, 500)
    y = np.sin(3 * x)
    car_y = np.sin(3 * position)

    ax.plot(x, y, color='black', linewidth=2)
    ax.scatter([position], [car_y], s=200, c='crimson', zorder=5)

    gx = goal_position
    gy = np.sin(3 * gx)
    ax.plot([gx, gx], [gy, gy + 0.2], color='black', linewidth=2)
    ax.fill([gx, gx + 0.06, gx], [gy + 0.2, gy + 0.17, gy + 0.14], color='gold')

    ax.set_xlim(min_position - 0.05, max_position + 0.05)
    ax.set_ylim(-1.25, 1.25)
    ax.set_title('TinySim Policy Playback')
    ax.grid(alpha=0.3)

    from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
    canvas = FigureCanvas(fig)
    canvas.draw()
    frame = np.asarray(canvas.buffer_rgba())[..., :3].copy()
    plt.close(fig)
    return frame


def load_agent_from_checkpoint(path: str | Path, device: str = DEVICE) -> DQNAgent:
    payload = torch.load(str(path), map_location=device)
    agent = make_agent_from_hparams(payload.get('hparams'), device=device)
    agent.load_payload(payload)
    return agent


def record_policy_frames(agent: DQNAgent, target_params: dict, max_steps: int = 200):
    sim_env = MountainCarEnv()
    sim_env.force = float(target_params['force'])
    sim_env.gravity = float(target_params['gravity'])

    state = sim_env.reset()
    obs = np.array([state['position'], state['velocity']], dtype=np.float32)

    frames = [render_tinysim_state_frame(state['position'], sim_env.min_position, sim_env.max_position, sim_env.goal_position)]

    done = bool(state['done'])
    steps = 0
    while (not done) and steps < max_steps:
        a = agent.act(obs, training=False)
        state = sim_env.step(a)
        obs = np.array([state['position'], state['velocity']], dtype=np.float32)
        done = bool(state['done'])
        steps += 1
        frames.append(render_tinysim_state_frame(state['position'], sim_env.min_position, sim_env.max_position, sim_env.goal_position))

    summary = {
        'solved': bool(done),
        'steps': int(steps),
        'final_position': float(state['position']),
    }
    return frames, summary


def show_animation(frames, interval=25):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis('off')
    img = ax.imshow(frames[0])

    def _update(frame):
        img.set_data(frame)
        return [img]

    ani = animation.FuncAnimation(fig, _update, frames=frames, interval=interval, blit=True)
    plt.close(fig)
    return HTML(ani.to_jshtml())


dr_best = load_agent_from_checkpoint(dr_result['best_checkpoint'], device=DEVICE)
bo_best = load_agent_from_checkpoint(bo_result['best_checkpoint'], device=DEVICE)

dr_frames, dr_summary = record_policy_frames(dr_best, TARGET_PARAMS, max_steps=TAU_STEPS)
bo_frames, bo_summary = record_policy_frames(bo_best, TARGET_PARAMS, max_steps=TAU_STEPS)

print('Pure DR summary:', dr_summary)
display(show_animation(dr_frames))

print('Pure BO summary:', bo_summary)
display(show_animation(bo_frames))